In [11]:
import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
import re
import janitor

In [3]:
race = pd.read_csv("data/philly_race_ethn_deceniall_p6/DECENNIALDHC2020.P9-Data.csv", skiprows=1)
race.head()

,Geography,Geographic Area Name,!!Total:,!!Total:!!Hispanic or Latino,!!Total:!!Not Hispanic or Latino:,!!Total:!!Not Hispanic or Latino:!!Population of one race:,!!Total:!!Not Hispanic or Latino:!!Population of one race:!!White alone,!!Total:!!Not Hispanic or Latino:!!Population of one race:!!Black or African American alone,!!Total:!!Not Hispanic or Latino:!!Population of one race:!!American Indian and Alaska Native alone,!!Total:!!Not Hispanic or Latino:!!Population of one race:!!Asian alone,...,!!Total:!!Not Hispanic or Latino:!!Population of two or more races:!!Population of five races:,!!Total:!!Not Hispanic or Latino:!!Population of two or more races:!!Population of five races:!!White; Black or African American; American Indian and Alaska Native; Asian; Native Hawaiian and Other Pacific Islander,!!Total:!!Not Hispanic or Latino:!!Population of two or more races:!!Population of five races:!!White; Black or African American; American Indian and Alaska Native; Asian; Some Other Race,!!Total:!!Not Hispanic or Latino:!!Population of two or more races:!!Population of five races:!!White; Black or African American; American Indian and Alaska Native; Native Hawaiian and Other Pacific Islander; Some Other Race,!!Total:!!Not Hispanic or Latino:!!Population of two or more races:!!Population of five races:!!White; Black or African American; Asian; Native Hawaiian and Other Pacific Islander; Some Other Race,!!Total:!!Not Hispanic or Latino:!!Population of two or more races:!!Population of five races:!!White; American Indian and Alaska Native; Asian; Native Hawaiian and Other Pacific Islander; Some Other Race,!!Total:!!Not Hispanic or Latino:!!Population of two or more races:!!Population of five races:!!Black or African American; American Indian and Alaska Native; Asian; Native Hawaiian and Other Pacific Islander; Some Other Race,!!Total:!!Not Hispanic or Latino:!!Population of two or more races:!!Population of six races:,!!Total:!!Not Hispanic or Latino:!!Population of two or more races:!!Population of six races:!!White; Black or African American; American Indian and Alaska Native; Asian; Native Hawaiian and Other Pacific Islander; Some Other Race,Unnamed: 75
0,1400000US42101000101,Census Tract 1.01; Philadelphia County; Pennsy...,2329,130,2199,2108,1791,105,4,200,...,0,0,0,0,0,0,0,0,0,NaN
1,1400000US42101000102,Census Tract 1.02; Philadelphia County; Pennsy...,3511,208,3303,3169,2719,132,6,286,...,4,4,0,0,0,0,0,0,0,NaN
2,1400000US42101000200,Census Tract 2; Philadelphia County; Pennsylvania,3367,125,3242,3125,784,251,7,2068,...,0,0,0,0,0,0,0,0,0,NaN
3,1400000US42101000300,Census Tract 3; Philadelphia County; Pennsylvania,4501,279,4222,4058,3100,300,0,641,...,0,0,0,0,0,0,0,0,0,NaN
4,1400000US42101000401,Census Tract 4.01; Philadelphia County; Pennsy...,3123,167,2956,2818,1754,249,5,796,...,0,0,0,0,0,0,0,0,0,NaN


In [6]:
race["TRACTCE"] = (
    race["Geographic Area Name"]
    .str.extract(r"Census Tract ([\d\.]+)")[0]   # extract tract number
    .str.replace(".", "", regex=False)           # remove decimal point
    .str.zfill(6)                                # pad to 6 digits
)


race['TRACTCE']

0      000101
1      000102
2      000002
3      000003
4      000401
        ...  
403    980905
404    980906
405    009891
406    009892
407    009893
Name: TRACTCE, Length: 408, dtype: object

In [20]:

def tract_to_tractce(geo_name):
    # extract the tract number from the string
    match = re.search(r'Census Tract ([\d.]+)', geo_name)
    if not match:
        return None
    
    tract = match.group(1)
    
    # split on decimal point
    if '.' in tract:
        integer_part, decimal_part = tract.split('.')
        # Pad decimal to 2 digits (e.g., "1" -> "01")
        decimal_part = decimal_part.ljust(2, '0')
    else:
        integer_part = tract
        decimal_part = '00'
    
    # combine and zero-pad to 6 digits
    tractce = f"{int(integer_part):04d}{decimal_part}"
    return tractce

race['TRACTCE'] = race['Geographic Area Name'].apply(tract_to_tractce).astype(str)
race['TRACTCE']

0      000101
1      000102
2      000200
3      000300
4      000401
        ...  
403    980905
404    980906
405    989100
406    989200
407    989300
Name: TRACTCE, Length: 408, dtype: object

In [21]:
# clean names
race_clean = race.clean_names()
race_clean.columns = race_clean.columns.str.replace("!!", "_", regex=False)
race_clean.columns = race_clean.columns.str.replace("__", "_", regex=False)
race_clean.columns = race_clean.columns.str.replace("_total_", "total_", regex=False)
race_clean.head()

,geography,geographic_area_name,total_,total_hispanic_or_latino,total_not_hispanic_or_latino_,total_not_hispanic_or_latino_population_of_one_race_,total_not_hispanic_or_latino_population_of_one_race_white_alone,total_not_hispanic_or_latino_population_of_one_race_black_or_african_american_alone,total_not_hispanic_or_latino_population_of_one_race_american_indian_and_alaska_native_alone,total_not_hispanic_or_latino_population_of_one_race_asian_alone,...,total_not_hispanic_or_latino_population_of_two_or_more_races_population_of_five_races_white;_black_or_african_american;_american_indian_and_alaska_native;_asian;_native_hawaiian_and_other_pacific_islander,total_not_hispanic_or_latino_population_of_two_or_more_races_population_of_five_races_white;_black_or_african_american;_american_indian_and_alaska_native;_asian;_some_other_race,total_not_hispanic_or_latino_population_of_two_or_more_races_population_of_five_races_white;_black_or_african_american;_american_indian_and_alaska_native;_native_hawaiian_and_other_pacific_islander;_some_other_race,total_not_hispanic_or_latino_population_of_two_or_more_races_population_of_five_races_white;_black_or_african_american;_asian;_native_hawaiian_and_other_pacific_islander;_some_other_race,total_not_hispanic_or_latino_population_of_two_or_more_races_population_of_five_races_white;_american_indian_and_alaska_native;_asian;_native_hawaiian_and_other_pacific_islander;_some_other_race,total_not_hispanic_or_latino_population_of_two_or_more_races_population_of_five_races_black_or_african_american;_american_indian_and_alaska_native;_asian;_native_hawaiian_and_other_pacific_islander;_some_other_race,total_not_hispanic_or_latino_population_of_two_or_more_races_population_of_six_races_,total_not_hispanic_or_latino_population_of_two_or_more_races_population_of_six_races_white;_black_or_african_american;_american_indian_and_alaska_native;_asian;_native_hawaiian_and_other_pacific_islander;_some_other_race,unnamed_75,tractce
0,1400000US42101000101,Census Tract 1.01; Philadelphia County; Pennsy...,2329,130,2199,2108,1791,105,4,200,...,0,0,0,0,0,0,0,0,NaN,000101
1,1400000US42101000102,Census Tract 1.02; Philadelphia County; Pennsy...,3511,208,3303,3169,2719,132,6,286,...,4,0,0,0,0,0,0,0,NaN,000102
2,1400000US42101000200,Census Tract 2; Philadelphia County; Pennsylvania,3367,125,3242,3125,784,251,7,2068,...,0,0,0,0,0,0,0,0,NaN,000200
3,1400000US42101000300,Census Tract 3; Philadelphia County; Pennsylvania,4501,279,4222,4058,3100,300,0,641,...,0,0,0,0,0,0,0,0,NaN,000300
4,1400000US42101000401,Census Tract 4.01; Philadelphia County; Pennsy...,3123,167,2956,2818,1754,249,5,796,...,0,0,0,0,0,0,0,0,NaN,000401


In [22]:

# create variables of total and proportion of at least part Black and Asian
black_cols = [col for col in race_clean.columns if "black_or_african_american" in col.lower()]
asian_cols = [col for col in race_clean.columns if "asian" in col.lower()]

race_clean["total_black_any"] = race_clean[black_cols].sum(axis=1)
race_clean["total_asian_any"] = race_clean[asian_cols].sum(axis=1)
race_clean['prop_black'] = race_clean['total_black_any']/race_clean['total_']
race_clean['prop_asian'] = race_clean['total_asian_any']/race_clean['total_']
race_clean['prop_hispanic_or_latino'] = race_clean['total_hispanic_or_latino']/race_clean['total_']
race_clean.head()

/var/folders/rz/h3gzrv2j09s6d_5vmn0bglw40000gn/T/ipykernel_57442/4117763509.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  race_clean["total_black_any"] = race_clean[black_cols].sum(axis=1)
/var/folders/rz/h3gzrv2j09s6d_5vmn0bglw40000gn/T/ipykernel_57442/4117763509.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  race_clean["total_asian_any"] = race_clean[asian_cols].sum(axis=1)
/var/folders/rz/h3gzrv2j09s6d_5vmn0bglw40000gn/T/ipykernel_57442/4117763509.py:7: SettingWithCopyWarning: 
A value is tryi

,geography,geographic_area_name,total_,total_hispanic_or_latino,total_not_hispanic_or_latino_,total_not_hispanic_or_latino_population_of_one_race_,total_not_hispanic_or_latino_population_of_one_race_white_alone,total_not_hispanic_or_latino_population_of_one_race_black_or_african_american_alone,total_not_hispanic_or_latino_population_of_one_race_american_indian_and_alaska_native_alone,total_not_hispanic_or_latino_population_of_one_race_asian_alone,...,total_not_hispanic_or_latino_population_of_two_or_more_races_population_of_five_races_black_or_african_american;_american_indian_and_alaska_native;_asian;_native_hawaiian_and_other_pacific_islander;_some_other_race,total_not_hispanic_or_latino_population_of_two_or_more_races_population_of_six_races_,total_not_hispanic_or_latino_population_of_two_or_more_races_population_of_six_races_white;_black_or_african_american;_american_indian_and_alaska_native;_asian;_native_hawaiian_and_other_pacific_islander;_some_other_race,unnamed_75,tractce,total_black_any,total_asian_any,prop_black,prop_asian,prop_hispanic_or_latino
0,1400000US42101000101,Census Tract 1.01; Philadelphia County; Pennsy...,2329,130,2199,2108,1791,105,4,200,...,0,0,0,NaN,000101,128,236,0.054959,0.101331,0.055818
1,1400000US42101000102,Census Tract 1.02; Philadelphia County; Pennsy...,3511,208,3303,3169,2719,132,6,286,...,0,0,0,NaN,000102,178,354,0.050698,0.100826,0.059242
2,1400000US42101000200,Census Tract 2; Philadelphia County; Pennsylvania,3367,125,3242,3125,784,251,7,2068,...,0,0,0,NaN,000200,286,2135,0.084942,0.634096,0.037125
3,1400000US42101000300,Census Tract 3; Philadelphia County; Pennsylvania,4501,279,4222,4058,3100,300,0,641,...,0,0,0,NaN,000300,362,714,0.080427,0.158631,0.061986
4,1400000US42101000401,Census Tract 4.01; Philadelphia County; Pennsy...,3123,167,2956,2818,1754,249,5,796,...,0,0,0,NaN,000401,280,865,0.089657,0.276977,0.053474


In [23]:
race_sub = race_clean[['geography', 'geographic_area_name', 'total_', 'tractce', 'prop_black', 'prop_asian', 'prop_hispanic_or_latino']]

race_sub = race_sub.rename(columns={"total_": "tot_pop", 
                                    "tractce": "TRACTCE"})

race_sub.head()

,geography,geographic_area_name,tot_pop,TRACTCE,prop_black,prop_asian,prop_hispanic_or_latino
0,1400000US42101000101,Census Tract 1.01; Philadelphia County; Pennsy...,2329,000101,0.054959,0.101331,0.055818
1,1400000US42101000102,Census Tract 1.02; Philadelphia County; Pennsy...,3511,000102,0.050698,0.100826,0.059242
2,1400000US42101000200,Census Tract 2; Philadelphia County; Pennsylvania,3367,000200,0.084942,0.634096,0.037125
3,1400000US42101000300,Census Tract 3; Philadelphia County; Pennsylvania,4501,000300,0.080427,0.158631,0.061986
4,1400000US42101000401,Census Tract 4.01; Philadelphia County; Pennsy...,3123,000401,0.089657,0.276977,0.053474


In [24]:
race_sub['prop_white_only'] = 1 - (race_sub['prop_black'] + race_sub['prop_asian'] + race_sub['prop_hispanic_or_latino'])
race_sub.head()

,geography,geographic_area_name,tot_pop,TRACTCE,prop_black,prop_asian,prop_hispanic_or_latino,prop_white_only
0,1400000US42101000101,Census Tract 1.01; Philadelphia County; Pennsy...,2329,000101,0.054959,0.101331,0.055818,0.787892
1,1400000US42101000102,Census Tract 1.02; Philadelphia County; Pennsy...,3511,000102,0.050698,0.100826,0.059242,0.789234
2,1400000US42101000200,Census Tract 2; Philadelphia County; Pennsylvania,3367,000200,0.084942,0.634096,0.037125,0.243837
3,1400000US42101000300,Census Tract 3; Philadelphia County; Pennsylvania,4501,000300,0.080427,0.158631,0.061986,0.698956
4,1400000US42101000401,Census Tract 4.01; Philadelphia County; Pennsy...,3123,000401,0.089657,0.276977,0.053474,0.579891


In [26]:
race_sub.dtypes

geography                   object
geographic_area_name        object
tot_pop                      int64
TRACTCE                     object
prop_black                 float64
prop_asian                 float64
prop_hispanic_or_latino    float64
prop_white_only            float64
dtype: object

In [25]:
race_sub.to_csv("modified_data/race_ethn_cleaned.csv")